# <img style="float: left; padding-right: 100px; width: 300px" src="../image/logo.png">AI4SG Bootcamp:



##  Combining datasets, Group by and Pivoting  operations


**Authors:** Faustine


---

In [ ]:
import numpy as np
import pandas as pd

# Concatenating data

The ``pd.concat`` function does all of the heavy lifting of combining data in different ways. ``pd.concat`` takes a list or dict of Series/DataFrame objects and concatenates them in a certain direction (`axis`) with some configurable handling of “what to do with the other axes”.


Assume we have the following different:

In [ ]:
data = {
    "country": [
        "Nigeria",
        "Rwanda",
        "Egypt",
        "Morocco",
    ],
    "population": [182.2, 11.3, 94.3, 34.4],
    "area": [923768, 26338, 1010408, 710850],
    "capital": ["Abuja", "Kigali", "Cairo", "Rabat"],
}
countries_africa = pd.DataFrame(data)
countries_africa

In [ ]:
data = {
    "country": ["Belgium", "France", "Germany", "Netherlands", "United Kingdom"],
    "population": [11.3, 64.3, 81.3, 16.9, 64.9],
    "area": [30510, 671308, 357050, 41526, 244820],
    "capital": ["Brussels", "Paris", "Berlin", "Amsterdam", "London"],
}
countries_europe = pd.DataFrame(data)
countries_europe

<div class="alert alert-success">
<b>Activity 1</b>:
Add a cloumn for pupulation density in each dataset 
</div>

### Combining rows - ``pd.concat``

We now want to combine the rows of both datasets:

In [ ]:
pd.concat([countries_europe, countries_africa])

When the two dataframes don't have the same set of columns, by default missing values get introduced:

In [ ]:
pd.concat([countries_europe, countries_africa[["country", "capital"]]], ignore_index=True, sort=False)

### Combining columns  - ``pd.concat`` with ``axis=1``

Assume we have another DataFrame for the same countries, but with some additional statistics:

In [ ]:
data = {"country": ["Belgium", "France", "Netherlands"], "GDP": [496477, 2650823, 820726], "area": [8.0, 9.9, 5.7]}
country_economics = pd.DataFrame(data).set_index("country")
country_economics

## Joining data with `pd.merge`
Using `pd.concat` above, we combined datasets that had the same columns or the same index values. But, another typical case if where you want to add information of second dataframe to a first one based on one of the columns. That can be done with [`pd.merge`](http://pandas.pydata.org/pandas-docs/stable/generated/pandas.DataFrame.merge.html).

Let's look again at the primary dataset, but taking a small subset of it to make the example easier to grasp:

In [ ]:
df = pd.read_csv("data/primary.csv")

In [ ]:
data = df.loc[:20, :]

In [ ]:
data

Assume we have another dataframe with more information about the 'REGION' locations:

In [ ]:
zones = pd.DataFrame(
    {
        "REGION": ["ARUSHA", "DAR ES SALAAM", "DODOMA", "GEITA"],
        "ZONES": ["Northern zone", "Costal zone", "Central zone", "Lake zone"],
        "COUNTRY": ["Tanzania", "Tanzania", "Tanzania", "Tanzania"],
    }
)

In [ ]:
zones

We now want to add those columns to the data dataframe, for which we can use `pd.merge`, specifying the column on which we want to merge the two datasets:

In [ ]:
pd.merge(data, zones, on="REGION", how="left")

In this case we use `how='left` (a "left join") because we wanted to keep the original rows of `df` and only add matching values from `zones` to it. Other options are 'inner', 'outer' and 'right' (see the [docs](http://pandas.pydata.org/pandas-docs/stable/merging.html#brief-primer-on-merge-methods-relational-algebra) for more on this).

## Group by" operations
Consider the following data

When analyzing data, you often calculate summary statistics (aggregations like the mean, max, ...). As we have seen before, we can easily calculate such a statistic for a Series or column using one of the many available methods. For example:

In [ ]:
df["FEMALE"].sum()

However, in many cases your data has certain groups in it, and in that case, you may want to calculate this statistic for each of the groups.

For example, in the above dataframe `df`,  REGION column has several possible values. When we want to calculate the sum for of FEMALE students in the three REGION, GEITA, DODOMA and ARUSHA, we could do the following:

In [ ]:
for key in ["GEITA", "DODOMA", "ARUSHA"]:
    print(key, df[df["REGION"] == key]["FEMALE"].sum())

This becomes very verbose when having multiple groups. You could make the above a bit easier by looping over the different values, but still, it is not very convenient to work with.

What we did above, applying a function on different groups, is a "groupby operation", and pandas provides some convenient functionality for this.



The "group by" concept: we want to **apply the same function on subsets of your dataframe, based on some key to split the dataframe in subsets**

This operation is also referred to as the "split-apply-combine" operation, involving the following steps:

* **Splitting** the data into groups based on some criteria
* **Applying** a function to each group independently
* **Combining** the results into a data structure

<img src="img/splitApplyCombine.png">

Similar to SQL `GROUP BY`

In [ ]:
df.groupby("REGION")[["FEMALE"]].sum()

In the previous example, we  grouped by a single column by passing its name. But, a column name is not the only value you can pass as the grouper in `df.groupby(grouper)`. Other possibilities for `grouper` are:

- a list of strings (to group by multiple columns)
- a Series (similar to a string indicating a column in df) or array
- function (to be applied on the index)
- levels=[], names of levels in a MultiIndex

In [ ]:
df.groupby(["REGION", "DISTRICT"])["FEMALE"].mean()

Oftentimes you want to know how many elements there are in a certain group (or in other words: the number of occurences of the different values from a column).

To get the size of the groups, we can use `size`:

In [ ]:
df.groupby("REGION").size()

## Pivoting data

People who know Excel, probably know the **Pivot** functionality:

In [ ]:
df = pd.DataFrame(
    {
        "Month": [
            "January",
            "January",
            "January",
            "January",
            "February",
            "February",
            "February",
            "February",
            "March",
            "March",
            "March",
            "March",
        ],
        "Category": [
            "Transportation",
            "Grocery",
            "Household",
            "Entertainment",
            "Transportation",
            "Grocery",
            "Household",
            "Entertainment",
            "Transportation",
            "Grocery",
            "Household",
            "Entertainment",
        ],
        "Amount": [74.0, 235.0, 175.0, 100.0, 115.0, 240.0, 225.0, 125.0, 90.0, 260.0, 200.0, 120.0],
    }
)

In [ ]:
df

In [ ]:
df.pivot(index="DISTRICT", columns="FEMALE", values="MALE")